# Day 1 - Data Serving

## From Collected Data to a Simple Dashboard

In the earlier notebooks, we collected data, cleaned it, saved it, and queried it.

Now we ask a new question:

```text
We already have data.
How do we serve it to a dashboard or an application?
```

In this notebook, we will:

1. Use a prepared DuckDB serving database.
2. Design simple queries for a dashboard.
3. Expose the data through a small API.
4. Run a local dashboard with `uvicorn`.

The prepared database is:

```text
day_1_weather_taxi_data.duckdb
```

Treat this database as the data interface for the dashboard. The collection code may change later, but the dashboard can stay stable if the served tables and views keep the same shape.

**Colab note:** The SQL and API-design parts can run in Colab if the DuckDB file is uploaded or mounted. The final `uvicorn` dashboard step should be run on a local computer because it starts a local web server.


## 1. What Is Data Serving?

Data serving means making processed data available to the people, tools, or systems that need it.

Common examples:

| Consumer | Data serving need |
|---|---|
| Dashboard | show latest values and charts |
| Analyst | query clean tables |
| Web app | request JSON from an API |
| Machine learning workflow | read feature tables |

For this lesson, we focus on a simple dashboard.

## 2. Serving Flow

A common data engineering flow is:

```text
Public APIs
   ->
Scheduled collection job
   ->
Serving database
   ->
Dashboard queries / API endpoints / frontend
```

The serving database should be easy to query for the application's needs.

## 3. Dashboard Requirements

Start with the product need, not the table.

Suppose we want a simple taxi and weather dashboard.

The dashboard needs to answer four questions:

1. What is the current taxi availability in each area?
2. What is the current weather in each area?
3. How did taxi availability change over time?
4. When was the data last updated?

These are intentionally simple requirements.

A key design point: these four questions do not require joining taxi and weather data into one table.

We can serve taxi data and weather data separately.

## 4. Where Did the Data Come From?

This data follows the same structure as the complete Day 1 pipeline.

The original data came from public APIs:

| Source | What it gives us |
|---|---|
| Taxi availability API | taxi locations at an API timestamp |
| Weather forecast API | area forecasts and valid time periods |

In the complete pipeline, taxi locations were aggregated into planning-area counts.

So instead of showing every taxi point, the dashboard can show:

```text
api_timestamp, planning_area, available_taxi_count
```

Weather forecasts are stored as:

```text
api_update_timestamp, valid_period_start, valid_period_end, area, forecast
```

A scheduled job collected these APIs repeatedly and wrote records into a database.

The `collection_runs` table records job history: source, scheduled time, start/end time, status, inserted row count, and message.

## 5. Install and Import Libraries

DuckDB is used for SQL queries.

FastAPI is introduced later for API serving.

In [5]:
# In Google Colab, run this cell if the packages are not installed.
#%pip install -q duckdb fastapi uvicorn

In [6]:
import duckdb
from pathlib import Path

## 6. Connect to the Serving Database

The path check below lets the notebook work from either the `day_1` folder or the parent `slides` folder.

In [7]:
def find_day1_base_dir() -> Path:
    current = Path.cwd().resolve()
    if (current / "day_1.pptx").exists() or (current / "shared_data").exists():
        return current
    for parent in current.parents:
        if parent.name == "day_1":
            return parent
    return current


base_dir = find_day1_base_dir()
db_candidates = [
    base_dir / "shared_data" / "day_1_weather_taxi_data.duckdb",
    base_dir / "day_1_weather_taxi_data.duckdb",
    Path("day_1_weather_taxi_data.duckdb"),
]
db_path = next((path for path in db_candidates if path.exists()), db_candidates[0])

print(db_path.resolve())
print("Database exists:", db_path.exists())


D:\SIT OneDrive\OneDrive - Singapore Institute Of Technology\project_IMDA\teaching\slides\day_1\shared_data\day_1_weather_taxi_data.duckdb
Database exists: True


In [8]:
con = duckdb.connect(str(db_path), read_only=True)

## 7. Review the Data We Have

First inspect the raw tables used by the dashboard.

The database may contain helper views, but in this lesson we reason from the raw table structures.


In [9]:
con.execute("SHOW TABLES").df()

,name
0,collection_runs
1,dashboard_area_snapshot
2,dim_area
3,fact_rainfall
4,fact_taxi
5,fact_weather
6,import_log
7,mini_area
8,mini_weather
9,rainfall_readings


In [10]:
con.execute("""
SELECT 'collection_runs' AS table_name, COUNT(*) AS row_count FROM collection_runs
UNION ALL
SELECT 'taxi_availability', COUNT(*) FROM taxi_availability
UNION ALL
SELECT 'weather_forecasts', COUNT(*) FROM weather_forecasts
""").df()


,table_name,row_count
0,collection_runs,12
1,taxi_availability,206
2,weather_forecasts,188


## 8. Raw Table Structures

These raw tables match the structure produced by the complete Day 1 pipeline.

Raw tables are good for audit and debugging.

In [11]:
con.execute("DESCRIBE taxi_availability").df()

,column_name,column_type,null,key,default,extra
0,id,BIGINT,NO,PRI,None,None
1,collection_time,VARCHAR,NO,None,None,None
2,api_timestamp,VARCHAR,NO,UNI,None,None
3,planning_area,VARCHAR,NO,UNI,None,None
4,available_taxi_count,BIGINT,NO,None,None,None
5,payload_hash,VARCHAR,NO,None,None,None
6,created_at,VARCHAR,NO,None,CAST(CURRENT_TIMESTAMP AS VARCHAR),None


In [12]:
con.execute("DESCRIBE weather_forecasts").df()

,column_name,column_type,null,key,default,extra
0,id,BIGINT,NO,PRI,None,None
1,collection_time,VARCHAR,NO,None,None,None
2,api_update_timestamp,VARCHAR,NO,UNI,None,None
3,valid_period_start,VARCHAR,NO,UNI,None,None
4,valid_period_end,VARCHAR,NO,UNI,None,None
5,area,VARCHAR,NO,UNI,None,None
6,forecast,VARCHAR,NO,None,None,None
7,payload_hash,VARCHAR,NO,None,None,None
8,created_at,VARCHAR,NO,None,CAST(CURRENT_TIMESTAMP AS VARCHAR),None


In [13]:
con.execute("DESCRIBE collection_runs").df()

,column_name,column_type,null,key,default,extra
0,id,BIGINT,NO,PRI,None,None
1,source,VARCHAR,NO,None,None,None
2,scheduled_for,VARCHAR,NO,None,None,None
3,started_at,VARCHAR,NO,None,None,None
4,ended_at,VARCHAR,NO,None,None,None
5,status,VARCHAR,NO,None,None,None
6,records_inserted,BIGINT,NO,None,0,None
7,message,VARCHAR,YES,None,None,None
8,created_at,VARCHAR,NO,None,CAST(CURRENT_TIMESTAMP AS VARCHAR),None


## 9. Preview Raw Data

Look at a few rows from each table.

In [14]:
con.execute("""
SELECT api_timestamp, planning_area, available_taxi_count
FROM taxi_availability
ORDER BY api_timestamp DESC, available_taxi_count DESC
LIMIT 10
""").df()

,api_timestamp,planning_area,available_taxi_count
0,2026-06-09T17:47:32+08:00,CHANGI,622
1,2026-06-09T17:47:32+08:00,ANG MO KIO,236
2,2026-06-09T17:47:32+08:00,TAMPINES,180
3,2026-06-09T17:47:32+08:00,BEDOK,178
4,2026-06-09T17:47:32+08:00,YISHUN,158
5,2026-06-09T17:47:32+08:00,WOODLANDS,145
6,2026-06-09T17:47:32+08:00,DOWNTOWN CORE,138
7,2026-06-09T17:47:32+08:00,TOA PAYOH,110
8,2026-06-09T17:47:32+08:00,HOUGANG,109
9,2026-06-09T17:47:32+08:00,JURONG WEST,108


In [15]:
con.execute("""
SELECT api_update_timestamp, valid_period_start, valid_period_end, area, forecast
FROM weather_forecasts
ORDER BY api_update_timestamp DESC, area
LIMIT 10
""").df()

,api_update_timestamp,valid_period_start,valid_period_end,area,forecast
0,2026-06-09T17:46:37+08:00,2026-06-09T17:30:00+08:00,2026-06-09T19:30:00+08:00,Ang Mo Kio,Light Rain
1,2026-06-09T17:46:37+08:00,2026-06-09T17:30:00+08:00,2026-06-09T19:30:00+08:00,Bedok,Moderate Rain
2,2026-06-09T17:46:37+08:00,2026-06-09T17:30:00+08:00,2026-06-09T19:30:00+08:00,Bishan,Light Rain
3,2026-06-09T17:46:37+08:00,2026-06-09T17:30:00+08:00,2026-06-09T19:30:00+08:00,Boon Lay,Moderate Rain
4,2026-06-09T17:46:37+08:00,2026-06-09T17:30:00+08:00,2026-06-09T19:30:00+08:00,Bukit Batok,Light Rain
5,2026-06-09T17:46:37+08:00,2026-06-09T17:30:00+08:00,2026-06-09T19:30:00+08:00,Bukit Merah,Moderate Rain
6,2026-06-09T17:46:37+08:00,2026-06-09T17:30:00+08:00,2026-06-09T19:30:00+08:00,Bukit Panjang,Light Rain
7,2026-06-09T17:46:37+08:00,2026-06-09T17:30:00+08:00,2026-06-09T19:30:00+08:00,Bukit Timah,Moderate Rain
8,2026-06-09T17:46:37+08:00,2026-06-09T17:30:00+08:00,2026-06-09T19:30:00+08:00,Central Water Catchment,Light Rain
9,2026-06-09T17:46:37+08:00,2026-06-09T17:30:00+08:00,2026-06-09T19:30:00+08:00,Changi,Light Rain


In [16]:
con.execute("""
SELECT source, scheduled_for, started_at, ended_at, status, records_inserted, message
FROM collection_runs
ORDER BY started_at DESC
LIMIT 10
""").df()

,source,scheduled_for,started_at,ended_at,status,records_inserted,message
0,rainfall,2026-06-09T17:47:57+08:00,2026-06-09T17:48:01+08:00,2026-06-09T17:48:03+08:00,success,74,Raw file: rainfall_20260609_174801.csv; rainfa...
1,weather,2026-06-09T17:47:57+08:00,2026-06-09T17:47:59+08:00,2026-06-09T17:48:01+08:00,success,47,Raw file: weather_forecast_20260609_174759.csv...
2,taxi,2026-06-09T17:47:57+08:00,2026-06-09T17:47:57+08:00,2026-06-09T17:47:59+08:00,success,53,Raw file: taxi_availability_20260609_174757.cs...
3,rainfall,2026-06-07T16:51:23+08:00,2026-06-07T16:51:23+08:00,2026-06-07T16:51:24+08:00,success,0,Raw file: rainfall_20260607_165123.csv; rainfa...
4,rainfall,2026-06-07T16:47:02+08:00,2026-06-07T16:47:02+08:00,2026-06-07T16:47:03+08:00,success,0,Raw file: rainfall_20260607_164702.csv; rainfa...
5,rainfall,2026-06-07T16:41:03+08:00,2026-06-07T16:41:06+08:00,2026-06-07T16:41:07+08:00,success,74,Raw file: rainfall_20260607_164106.csv; rainfa...
6,weather,2026-06-07T16:41:03+08:00,2026-06-07T16:41:05+08:00,2026-06-07T16:41:06+08:00,success,47,Raw file: weather_forecast_20260607_164105.csv...
7,taxi,2026-06-07T16:41:03+08:00,2026-06-07T16:41:03+08:00,2026-06-07T16:41:05+08:00,success,50,Raw file: taxi_availability_20260607_164103.cs...
8,weather,2026-05-30T17:36:19+08:00,2026-05-30T17:36:23+08:00,2026-05-30T17:36:24+08:00,success,47,Raw file: weather_forecast_20260530_173623.csv...
9,taxi,2026-05-30T17:36:19+08:00,2026-05-30T17:36:19+08:00,2026-05-30T17:36:23+08:00,success,52,Raw file: taxi_availability_20260530_173619.cs...


## 10. Analyze the Serving Requirements

From here, think like a data engineer designing a serving layer.

For every dashboard requirement, ask:

1. What question does the user want to answer?
2. Which raw table has the data?
3. What input parameter do we need?
4. What SQL query answers the question?
5. What columns should the API return?

This is the bridge from SQL to data serving.

## 11. Requirement 1: Current Taxi Availability

Question:

```text
What is the current taxi availability in each area?
```

Raw table:

```text
taxi_availability
```

Important columns:

| Column | Meaning |
|---|---|
| `api_timestamp` | timestamp of the taxi API snapshot |
| `planning_area` | Singapore planning area |
| `available_taxi_count` | number of available taxis in that area |

For teaching, assume we already know the timestamp we want.

So the SQL idea is simple:

```text
Find all taxi rows where api_timestamp equals the selected timestamp.
```

In [17]:
known_taxi_timestamp = con.execute("""
SELECT MAX(api_timestamp) AS latest_taxi_timestamp
FROM taxi_availability
""").fetchone()[0]

known_taxi_timestamp

'2026-06-09T17:47:32+08:00'

In [18]:
con.execute("""
SELECT
    api_timestamp,
    planning_area,
    available_taxi_count
FROM taxi_availability
WHERE api_timestamp = ?
ORDER BY available_taxi_count DESC, planning_area
""", [known_taxi_timestamp]).df()

,api_timestamp,planning_area,available_taxi_count
0,2026-06-09T17:47:32+08:00,CHANGI,622
1,2026-06-09T17:47:32+08:00,ANG MO KIO,236
2,2026-06-09T17:47:32+08:00,TAMPINES,180
3,2026-06-09T17:47:32+08:00,BEDOK,178
4,2026-06-09T17:47:32+08:00,YISHUN,158
5,2026-06-09T17:47:32+08:00,WOODLANDS,145
6,2026-06-09T17:47:32+08:00,DOWNTOWN CORE,138
7,2026-06-09T17:47:32+08:00,TOA PAYOH,110
8,2026-06-09T17:47:32+08:00,HOUGANG,109
9,2026-06-09T17:47:32+08:00,JURONG WEST,108


API response shape:

```text
api_timestamp, planning_area, available_taxi_count
```

Possible endpoint:

```text
GET /api/taxi/current?timestamp=2026-05-28T22:39:45+08:00
```

## 12. Requirement 2: Current Weather

Question:

```text
What is the current weather in each area?
```

Raw table:

```text
weather_forecasts
```

Important columns:

| Column | Meaning |
|---|---|
| `api_update_timestamp` | when the weather API forecast was updated |
| `valid_period_start` | forecast start time |
| `valid_period_end` | forecast end time |
| `area` | forecast area |
| `forecast` | weather forecast text |

Again, assume we already know the weather API update timestamp.

The SQL idea:

```text
Find all weather rows where api_update_timestamp equals the selected update timestamp.
```

In [19]:
known_weather_update = con.execute("""
SELECT MAX(api_update_timestamp) AS latest_weather_update
FROM weather_forecasts
""").fetchone()[0]

known_weather_update

'2026-06-09T17:46:37+08:00'

In [20]:
con.execute("""
SELECT
    api_update_timestamp,
    area,
    forecast,
    valid_period_start,
    valid_period_end
FROM weather_forecasts
WHERE api_update_timestamp = ?
ORDER BY area
""", [known_weather_update]).df()

,api_update_timestamp,area,forecast,valid_period_start,valid_period_end
0,2026-06-09T17:46:37+08:00,Ang Mo Kio,Light Rain,2026-06-09T17:30:00+08:00,2026-06-09T19:30:00+08:00
1,2026-06-09T17:46:37+08:00,Bedok,Moderate Rain,2026-06-09T17:30:00+08:00,2026-06-09T19:30:00+08:00
2,2026-06-09T17:46:37+08:00,Bishan,Light Rain,2026-06-09T17:30:00+08:00,2026-06-09T19:30:00+08:00
3,2026-06-09T17:46:37+08:00,Boon Lay,Moderate Rain,2026-06-09T17:30:00+08:00,2026-06-09T19:30:00+08:00
4,2026-06-09T17:46:37+08:00,Bukit Batok,Light Rain,2026-06-09T17:30:00+08:00,2026-06-09T19:30:00+08:00
5,2026-06-09T17:46:37+08:00,Bukit Merah,Moderate Rain,2026-06-09T17:30:00+08:00,2026-06-09T19:30:00+08:00
6,2026-06-09T17:46:37+08:00,Bukit Panjang,Light Rain,2026-06-09T17:30:00+08:00,2026-06-09T19:30:00+08:00
7,2026-06-09T17:46:37+08:00,Bukit Timah,Moderate Rain,2026-06-09T17:30:00+08:00,2026-06-09T19:30:00+08:00
8,2026-06-09T17:46:37+08:00,Central Water Catchment,Light Rain,2026-06-09T17:30:00+08:00,2026-06-09T19:30:00+08:00
9,2026-06-09T17:46:37+08:00,Changi,Light Rain,2026-06-09T17:30:00+08:00,2026-06-09T19:30:00+08:00


API response shape:

```text
api_update_timestamp, area, forecast, valid_period_start, valid_period_end
```

Possible endpoint:

```text
GET /api/weather/current?update_timestamp=2026-05-28T22:06:32+08:00
```

## 13. Requirement 3: Taxi Availability Over Time

Question:

```text
How did taxi availability change over time?
```

For the dashboard, we want four columns:

```text
api_timestamp, forecast, available_taxi_count, planning_area
```

This one is slightly more interesting.

Taxi data is collected roughly every 10 minutes.

Weather forecast data is updated less frequently, and each forecast has a valid time period.

So for each taxi timestamp, we need to attach the weather forecast that was valid at that time.

SQL idea:

1. Filter taxi rows for one planning area.
2. Match weather rows where the area is the same.
3. Keep weather rows whose valid period contains the taxi timestamp.
4. If multiple weather rows match, choose the latest weather update available at that taxi timestamp.

This is a join, but now the join is required by the product question.

In [21]:
area_name = "ANG MO KIO"

con.execute("""
WITH taxi_series AS (
    SELECT
        id,
        api_timestamp,
        planning_area,
        available_taxi_count
    FROM taxi_availability
    WHERE planning_area = ?
), weather_matches AS (
    SELECT
        t.api_timestamp,
        w.forecast,
        t.available_taxi_count,
        t.planning_area,
        ROW_NUMBER() OVER (
            PARTITION BY t.id
            ORDER BY w.api_update_timestamp DESC NULLS LAST
        ) AS match_rank
    FROM taxi_series AS t
    LEFT JOIN weather_forecasts AS w
    ON UPPER(TRIM(t.planning_area)) = UPPER(TRIM(w.area))
    AND t.api_timestamp >= w.valid_period_start
    AND t.api_timestamp < w.valid_period_end
    AND w.api_update_timestamp <= t.api_timestamp
)
SELECT
    api_timestamp,
    forecast,
    available_taxi_count,
    planning_area
FROM weather_matches
WHERE match_rank = 1
ORDER BY api_timestamp
LIMIT 50
""", [area_name]).df()

,api_timestamp,forecast,available_taxi_count,planning_area
0,2026-05-30T14:28:54+08:00,Cloudy,84,ANG MO KIO
1,2026-05-30T17:36:02+08:00,Cloudy,94,ANG MO KIO
2,2026-06-07T16:40:30+08:00,None,91,ANG MO KIO
3,2026-06-09T17:47:32+08:00,Light Rain,236,ANG MO KIO


Handling the 10-minute vs 30-minute issue:

- Taxi snapshots are more frequent.
- Weather rows cover a valid period.
- We do not need to force weather data into 10-minute intervals manually.
- Instead, each taxi row looks up the weather forecast that was valid at that taxi timestamp.

This is similar to saying:

```text
For this taxi timestamp, what weather forecast was active?
```

API response shape:

```text
api_timestamp, forecast, available_taxi_count, planning_area
```

Possible endpoint:

```text
GET /api/taxi/timeseries?area=ANG%20MO%20KIO
```

## 14. Requirement 4: Last Updated Time

Question:

```text
When was the data last updated?
```

This is a freshness question.

For taxi data, use the latest `api_timestamp`.

For weather data, use the latest `api_update_timestamp`.

The `collection_runs` table can also tell us when scheduled jobs last ran.

In [22]:
con.execute("""
SELECT
    'taxi' AS source,
    MAX(api_timestamp) AS latest_data_timestamp,
    COUNT(*) AS row_count
FROM taxi_availability
UNION ALL
SELECT
    'weather' AS source,
    MAX(api_update_timestamp) AS latest_data_timestamp,
    COUNT(*) AS row_count
FROM weather_forecasts
ORDER BY source
""").df()

,source,latest_data_timestamp,row_count
0,taxi,2026-06-09T17:47:32+08:00,206
1,weather,2026-06-09T17:46:37+08:00,188


In [23]:
con.execute("""
SELECT
    source,
    status,
    COUNT(*) AS num_runs,
    SUM(records_inserted) AS total_records_inserted,
    MAX(started_at) AS latest_run
FROM collection_runs
GROUP BY source, status
ORDER BY source, status
""").df()

,source,status,num_runs,total_records_inserted,latest_run
0,rainfall,success,4,148.0,2026-06-09T17:48:01+08:00
1,taxi,success,4,206.0,2026-06-09T17:47:57+08:00
2,weather,success,4,188.0,2026-06-09T17:47:59+08:00


API response shape:

```text
source, latest_data_timestamp, row_count
```

Possible endpoint:

```text
GET /api/freshness
```

## 15. API Design Methodology

Before writing FastAPI code, design the API.

A useful method:

1. Start from dashboard questions.
2. Convert each question into one endpoint.
3. Decide what parameters the endpoint needs.
4. Decide the response columns.
5. Keep endpoint names stable.
6. Keep SQL inside the API layer so the browser receives clean JSON.

Good API design is not just writing code.

It is deciding a clear contract between the browser page and the API server.

## 16. API Design for This Dashboard

| Dashboard question | Endpoint | Parameters | Returns |
|---|---|---|---|
| Current taxi availability | `GET /api/taxi/current` | optional `timestamp` | `api_timestamp`, `planning_area`, `available_taxi_count` |
| Current weather | `GET /api/weather/current` | optional `update_timestamp` | `api_update_timestamp`, `area`, `forecast`, `valid_period_start`, `valid_period_end` |
| Taxi availability over time | `GET /api/taxi/timeseries` | `area` | `api_timestamp`, `forecast`, `available_taxi_count`, `planning_area` |
| Last updated time | `GET /api/freshness` | none | `source`, `latest_data_timestamp`, `row_count` |

Optional parameters can default to the latest available timestamp.

That makes the frontend simpler.

In [24]:
current_taxi_sql = """
SELECT
    api_timestamp,
    planning_area,
    available_taxi_count
FROM taxi_availability_typed
WHERE api_timestamp = COALESCE(
    CAST(? AS TIMESTAMPTZ),
    (SELECT MAX(api_timestamp) FROM taxi_availability_typed)
)
ORDER BY available_taxi_count DESC, planning_area
"""

current_weather_sql = """
SELECT
    api_update_timestamp,
    area,
    forecast,
    valid_period_start,
    valid_period_end
FROM weather_forecasts_typed
WHERE api_update_timestamp = COALESCE(
    CAST(? AS TIMESTAMPTZ),
    (SELECT MAX(api_update_timestamp) FROM weather_forecasts_typed)
)
ORDER BY area
"""

taxi_timeseries_sql = """
WITH taxi_series AS (
    SELECT
        id,
        api_timestamp,
        planning_area,
        available_taxi_count
    FROM taxi_availability_typed
    WHERE planning_area = ?
), weather_matches AS (
    SELECT
        t.api_timestamp,
        w.forecast,
        t.available_taxi_count,
        t.planning_area,
        ROW_NUMBER() OVER (
            PARTITION BY t.id
            ORDER BY w.api_update_timestamp DESC NULLS LAST
        ) AS match_rank
    FROM taxi_series AS t
    LEFT JOIN weather_forecasts_typed AS w
    ON UPPER(TRIM(t.planning_area)) = UPPER(TRIM(w.area))
    AND t.api_timestamp >= w.valid_period_start
    AND t.api_timestamp < w.valid_period_end
    AND w.api_update_timestamp <= t.api_timestamp
)
SELECT
    api_timestamp,
    forecast,
    available_taxi_count,
    planning_area
FROM weather_matches
WHERE match_rank = 1
ORDER BY api_timestamp
"""

freshness_sql = """
SELECT
    'taxi' AS source,
    MAX(api_timestamp) AS latest_data_timestamp,
    COUNT(*) AS row_count
FROM taxi_availability_typed
UNION ALL
SELECT
    'weather' AS source,
    MAX(api_update_timestamp) AS latest_data_timestamp,
    COUNT(*) AS row_count
FROM weather_forecasts_typed
ORDER BY source
"""

## 17. FastAPI Skeleton

FastAPI turns these SQL queries into HTTP endpoints.

`@app.get("/some/url")` is a route decorator.

It tells FastAPI: when the browser sends a GET request to this URL, run the Python function immediately below it.

Function arguments become query parameters, and the returned Python list/dict is converted to JSON for the browser.

The frontend does not need to know SQL.

It only calls URLs and receives JSON.

In [25]:
from fastapi import FastAPI

app = FastAPI(title="Taxi Weather Dashboard API")


def query_duckdb(sql: str, params=None):
    with duckdb.connect(str(db_path), read_only=True) as api_con:
        return api_con.execute(sql, params or []).df().to_dict(orient="records")

In [26]:
# Register a GET endpoint at /api/taxi/current.
# When the browser requests this URL, FastAPI calls api_current_taxi().
@app.get("/api/taxi/current")
# timestamp is optional. If the URL includes ?timestamp=..., FastAPI passes it in.
# If it is missing, timestamp becomes None and the SQL falls back to the latest row.
def api_current_taxi(timestamp: str | None = None):
    # Returning a list of dictionaries is enough; FastAPI serializes it as JSON.
    return query_duckdb(current_taxi_sql, [timestamp])

In [27]:
# This endpoint follows the same pattern for weather data.
# URL: /api/weather/current?update_timestamp=...
@app.get("/api/weather/current")
def api_current_weather(update_timestamp: str | None = None):
    return query_duckdb(current_weather_sql, [update_timestamp])

In [28]:
# FastAPI maps the query parameter named area to this Python argument.
# Example URL: /api/taxi/timeseries?area=BEDOK
@app.get("/api/taxi/timeseries")
def api_taxi_timeseries(area: str = "ANG MO KIO"):
    # The SQL uses a parameter placeholder, so the user input is passed safely.
    return query_duckdb(taxi_timeseries_sql, [area.upper()])

In [29]:
# This endpoint has no query parameters.
# A GET request to /api/freshness simply runs api_freshness().
@app.get("/api/freshness")
def api_freshness():
    return query_duckdb(freshness_sql)

## 18. Dashboard Frontend: Calling the API

The frontend calls the API endpoints and renders the JSON response.

The browser does not run SQL directly.

Frontend responsibility:

1. call endpoint,
2. receive JSON,
3. render table/chart,
4. refresh when user changes input.

In [30]:
dashboard_html = """<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1" />
  <title>Taxi Weather Dashboard</title>
  <style>
    body { font-family: Arial, sans-serif; margin: 24px; background: #f7f7f8; color: #222; }
    header { margin-bottom: 20px; }
    .grid { display: grid; grid-template-columns: repeat(2, minmax(0, 1fr)); gap: 16px; }
    section { background: white; border: 1px solid #ddd; border-radius: 8px; padding: 16px; }
    table { width: 100%; border-collapse: collapse; font-size: 14px; }
    th, td { border-bottom: 1px solid #eee; padding: 8px; text-align: left; }
    th { background: #fafafa; }
    input, button { padding: 8px; font-size: 14px; }
    @media (max-width: 800px) { .grid { grid-template-columns: 1fr; } }
  </style>
</head>
<body>
  <header>
    <h1>Taxi Weather Dashboard</h1>
    <p>Current taxi availability, current weather, taxi trend, and data freshness.</p>
  </header>

  <div class="grid">
    <section>
      <h2>Current Taxi Availability</h2>
      <table id="taxi-table"></table>
    </section>

    <section>
      <h2>Current Weather</h2>
      <table id="weather-table"></table>
    </section>

    <section>
      <h2>Taxi Availability Over Time</h2>
      <input id="area-input" value="ANG MO KIO" />
      <button onclick="loadTimeseries()">Load</button>
      <table id="timeseries-table"></table>
    </section>

    <section>
      <h2>Last Updated</h2>
      <table id="freshness-table"></table>
    </section>
  </div>

  <script>
    function renderTable(id, rows) {
      const table = document.getElementById(id);
      if (!rows.length) {
        table.innerHTML = '<tr><td>No data</td></tr>';
        return;
      }
      const columns = Object.keys(rows[0]);
      const header = '<tr>' + columns.map(c => `<th>${c}</th>`).join('') + '</tr>';
      const body = rows.map(row => '<tr>' + columns.map(c => `<td>${row[c] ?? ''}</td>`).join('') + '</tr>').join('');
      table.innerHTML = header + body;
    }

    async function loadJson(url) {
      const response = await fetch(url);
      return response.json();
    }

    async function loadDashboard() {
      renderTable('taxi-table', await loadJson('/api/taxi/current'));
      renderTable('weather-table', await loadJson('/api/weather/current'));
      renderTable('freshness-table', await loadJson('/api/freshness'));
      await loadTimeseries();
    }

    async function loadTimeseries() {
      const area = document.getElementById('area-input').value;
      const rows = await loadJson(`/api/taxi/timeseries?area=${encodeURIComponent(area)}`);
      renderTable('timeseries-table', rows.slice(-20));
    }

    loadDashboard();
  </script>
</body>
</html>
"""

Path("dashboard_frontend.html").write_text(dashboard_html, encoding="utf-8")
print("Saved dashboard_frontend.html")

Saved dashboard_frontend.html


## 19. Run This as a Real Web Page

The notebook explains the idea, but a browser needs a running web server.

To make the dashboard real, keep these files in the same folder:

| File | Purpose |
|---|---|
| `shared_data/day_1_weather_taxi_data.duckdb` | project DuckDB database used by storage and serving |
| `serving_api.py` | FastAPI API server that queries DuckDB |
| `dashboard_frontend.html` | browser page that calls the API |

This is the practical workflow:

1. Design the dashboard requirements.
2. Write and test the SQL queries in the notebook.
3. Put the API code into `serving_api.py`.
4. Put the HTML/JavaScript into `dashboard_frontend.html`.
5. Run FastAPI with `uvicorn`.
6. Open the browser.

From the `day_1/04_serving_dashboard` folder, run:

```bash
python -m uvicorn serving_api:app --reload --host 127.0.0.1 --port 8000
```

Then open:

```text
http://127.0.0.1:8000
```

API documentation is available at:

```text
http://127.0.0.1:8000/docs
```

If port `8000` is already used, try another port:

```bash
python -m uvicorn serving_api:app --reload --host 127.0.0.1 --port 8001
```

## 20. How to Use AI Without Losing the Design

AI can help write the FastAPI file and the HTML page.

But you should still provide the data contract.

A weak prompt is:

```text
Make me a dashboard.
```

A better prompt is:

```text
I have a DuckDB database named shared_data/day_1_weather_taxi_data.duckdb.
It has these tables:

1. taxi_availability_typed
   - api_timestamp
   - planning_area
   - available_taxi_count

2. weather_forecasts_typed
   - api_update_timestamp
   - valid_period_start
   - valid_period_end
   - area
   - forecast

Build a FastAPI API server and a simple HTML frontend.

The dashboard needs four features:

1. Current taxi availability in each area.
   Endpoint: GET /api/taxi/current
   Return: api_timestamp, planning_area, available_taxi_count.
   If no timestamp is provided, use the latest api_timestamp.

2. Current weather in each area.
   Endpoint: GET /api/weather/current
   Return: api_update_timestamp, area, forecast, valid_period_start, valid_period_end.
   If no update_timestamp is provided, use the latest api_update_timestamp.

3. Taxi availability over time for one planning area.
   Endpoint: GET /api/taxi/timeseries?area=ANG%20MO%20KIO
   Return: api_timestamp, forecast, available_taxi_count, planning_area.
   Taxi data is about every 10 minutes. Weather forecasts cover valid periods.
   Match each taxi timestamp to the weather row where:
   - same area after upper/trim normalization
   - taxi api_timestamp is within valid_period_start and valid_period_end
   - weather api_update_timestamp is not later than the taxi timestamp
   If multiple weather rows match, use the latest api_update_timestamp.

4. Data freshness.
   Endpoint: GET /api/freshness
   Return: source, latest_data_timestamp, row_count.

Create:
- serving_api.py
- dashboard_frontend.html

The frontend should call these API endpoints with fetch() and render the JSON in tables.
```

Why this works better:

- It gives the database name.
- It gives the table names and columns.
- It gives endpoint names.
- It gives response shapes.
- It explains the time-matching rule.

AI writes better code when the human gives a clear contract.

## 21. What You Should Be Able to Explain

After building the page, you should be able to explain:

1. Which SQL query answers each dashboard question.
2. Which API endpoint serves each query.
3. What JSON shape each endpoint returns.
4. How the frontend calls the endpoint using `fetch()`.
5. Why the time series endpoint needs weather matching logic.
6. How AI helped, and what design decisions still came from you.

## 22. Data Serving Checklist

Before building the app, check:

1. What dashboard question are we answering?
2. Which raw table has the data?
3. What input parameter is needed?
4. What columns should the API return?
5. Is a join actually required?
6. How do we handle different update frequencies?
7. Is freshness visible to users?
8. Can the frontend consume the response easily?

## 23. Short Quiz

1. Why do we start from dashboard requirements instead of database tables?
2. Why does taxi availability over time need weather matching logic?
3. What does FastAPI add to the serving layer?

## 24. Suggested Answers

1. Requirements tell us what data shape users actually need.
2. Taxi data is more frequent than weather data, so each taxi timestamp needs the forecast valid at that time.
3. FastAPI exposes SQL query results as HTTP/JSON endpoints that a frontend can call.

## 25. Summary

In this notebook, we learned:

1. Data serving starts from user-facing requirements.
2. Current taxi availability can be served by filtering one taxi snapshot timestamp.
3. Current weather can be served by filtering one weather update timestamp.
4. Taxi availability over time can include weather by matching each taxi timestamp to a valid forecast period.
5. API design defines a contract between the API server and the browser page.
6. FastAPI can expose DuckDB query results as JSON.
7. A frontend calls API endpoints and renders the response.
8. You can run the page locally with `uvicorn`.
9. AI can generate app code more reliably after the data contract is clear.
